In [ ]:
import glob
import re
import matplotlib.pyplot as plt
import os

output_folder = "./results"

def _extract_metric(line, keys):
    for key in keys:
        m = re.search(rf"{key}\s*[:=]\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)", line, flags=re.IGNORECASE)
        if m:
            return float(m.group(1))
    return None


def load_training_curve(log_file):

    epochs, valid_pesq, valid_csig, valid_cbak, valid_covl = [], [], [], [], []

    with open(log_file, "r", encoding="utf-8") as f:
        for line in f:
            if "epoch" not in line.lower():
                continue

            m_epoch = re.search(r"epoch\s*[:=]\s*(\d+)", line, flags=re.IGNORECASE)
            if not m_epoch:
                continue

            epochs.append(int(m_epoch.group(1)))
            valid_pesq.append(_extract_metric(line, [r"pesq", r"valid[_\s-]*pesq"]))
            valid_csig.append(_extract_metric(line, [r"csig", r"valid[_\s-]*csig"]))
            valid_cbak.append(_extract_metric(line, [r"cbak", r"valid[_\s-]*cbak"]))
            valid_covl.append(_extract_metric(line, [r"covl", r"valid[_\s-]*covl"]))

    return {
        "epoch": epochs,
        "valid_pesq": valid_pesq,
        "valid_csig": valid_csig,
        "valid_cbak": valid_cbak,
        "valid_covl": valid_covl,
    }


candidates = sorted(glob.glob(f"{output_folder}/**/train_log.txt", recursive=True))
if not candidates:
    raise FileNotFoundError("Kein train_log.txt gefunden. Trainiere zuerst mindestens 1 Run.")

log_file = candidates[-1]
curves = load_training_curve(log_file)

if len(curves["epoch"]) == 0:
    raise RuntimeError(f"Keine Epochenzeilen in {log_file} gefunden.")

fig, axes = plt.subplots(2, 2, figsize=(12, 8), dpi=120)
plots = [
    ("valid_pesq", "Valid PESQ"),
    ("valid_csig", "Valid CSIG"),
    ("valid_cbak", "Valid CBAK"),
    ("valid_covl", "Valid COVL"),
]

for ax, (key, title) in zip(axes.flat, plots):
    y = curves[key]
    x = [e for e, v in zip(curves["epoch"], y) if v is not None]
    y = [v for v in y if v is not None]

    if len(x) > 0:
        ax.plot(x, y, marker="o", linewidth=1.5, markersize=3)
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)

plt.suptitle(f"Training Curves ({os.path.dirname(log_file)})")
plt.tight_layout()

os.makedirs(figures_folder, exist_ok=True)
out_png = os.path.join(figures_folder, "training_curves_latest.png")
plt.savefig(out_png, bbox_inches="tight")
plt.show()

print(f"Log-Datei: {log_file}")
print(f"Diagramm gespeichert: {out_png}")

In [ ]:
import re
import glob
import os
import matplotlib.pyplot as plt

%matplotlib inline

log_files = glob.glob("./**/train_log.txt", recursive=True)

if not log_files:
    print("Keine train_log.txt Dateien gefunden.")
else:
    pd_loss_pattern = re.compile(r"PD loss \(unscaled\):\s*([-\d.e+]+)")
    epoch_pattern   = re.compile(r"Epoch:\s*(\d+)")

    for log_path in log_files:
        epochs_losses = {}
        current_losses = []

        with open(log_path, "r") as f:
            for line in f:
                pd_match    = pd_loss_pattern.search(line)
                epoch_match = epoch_pattern.search(line)

                if pd_match:
                    current_losses.append(float(pd_match.group(1)))
                elif epoch_match:
                    epoch_num = int(epoch_match.group(1))
                    if current_losses:
                        epochs_losses[epoch_num] = current_losses[:]
                    current_losses = []

        if current_losses:
            next_epoch = max(epochs_losses.keys(), default=0) + 1
            epochs_losses[next_epoch] = current_losses

        if not epochs_losses:
            print(f"Keine Daten gefunden in: {log_path}")
            continue

        epochs   = sorted(epochs_losses.keys())
        min_vals = [max(epochs_losses[e]) for e in epochs]

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(epochs, min_vals, marker="o", linewidth=1.8, markersize=6,
                color="#e05c5c", markerfacecolor="white", markeredgewidth=2)

        for e, v in zip(epochs, min_vals):
            ax.annotate(f"{v:.4f}", xy=(e, v), xytext=(0, 8),
                        textcoords="offset points", ha="center", fontsize=8, color="#555")

        run_name = os.path.basename(os.path.dirname(log_path))
        ax.set_xlabel("Epoche", fontsize=11)
        ax.set_ylabel("Min. PD Loss (unscaled)", fontsize=11)
        ax.set_title(f"Maximaler Difference Loss pro Epoche\n{run_name}", fontsize=13, fontweight="bold")
        for e in range(0, max(epochs) + 1, 10):
            ax.axvline(x=e, color="red", linewidth=1.2, alpha=0.6)

        ax.set_xticks(epochs)
        ax.grid(linestyle="--", alpha=0.4)
        fig.tight_layout()
        plt.show()